# Data Visualization — Quiz (Lectures 1–7)

**Time:** 90 minutes
**Tools:** Plotly only (`plotly.express` first, then `update_traces` / `update_layout` / `update_xaxes` / `update_yaxes`)

**Your task:** answer each analytical question with **one polished, explanatory visualization**. For every question **you decide which chart type fits best** — and justify the choice in one or two sentences.

**Every figure must have:**
1. An **insight title** that states the finding, not just the variables
2. `template='plotly_white'` and a decluttered layout (no unnecessary gridlines, legends or axis titles)
3. A **CVD-safe palette** — no red/green; grey for context + one accent colour where it helps
4. Colour that **encodes meaning**, not decoration
5. Direct labels or annotations where they help the reader

**Chart types covered in the course** — choose from these:
line chart · sorted bar chart · stacked / 100% stacked bar chart · scatter plot · bubble chart · histogram · box plot · violin plot · choropleth map · treemap

> 💡 Each question has one chart type that fits it best, and the questions together cover most of the list above. If two of your answers use the same chart type, check the question again.


**Submit:** this notebook pushed to your personal repo as `quiz/dataviz_quiz.ipynb`

## The dataset — World Population 1970–2022

**Source:** [World Population Dataset](https://www.kaggle.com/datasets/iamsouravbanerjee/world-population-dataset) on Kaggle, compiled from Worldometer (UN World Population Prospects estimates). **234 rows × 16 columns — one row per country or territory, no missing values.**

| Column | Type | Description |
|---|---|---|
| `Country`, `Capital` | categorical | Country / territory and its capital |
| `Continent` | categorical | Africa, Asia, Europe, North America, Oceania, South America |
| `CCA3` | spatial | ISO-3 country code — use with `locationmode='ISO-3'` |
| `1970 Population` … `2022 Population` | temporal | Population in 1970, 1980, 1990, 2000, 2010, 2015, 2020, 2022 |
| `Area` | numerical | Land area in km² |
| `Density` | numerical | People per km² (2022) |
| `World share (%)` | numerical | Share of world population in 2022 |
| `Rank` | numerical | Rank by 2022 population |

**The next setup cell also gives you `df_long` — the same data in **tidy (long) format** with one row per country per year. Use `df` for one-year questions and `df_long` for questions about change over time**

In [15]:
import pandas as pd
import plotly.express as px
import numpy as np

df = pd.read_csv('/content/world_population.csv')

# Shorter column names
df = df.rename(columns={'Country/Territory': 'Country', 'Area (km²)': 'Area',
                        'Density (per km²)': 'Density', 'World Population Percentage': 'World share (%)'})

# Tidy (long) version: one row per country per year
year_cols = [c for c in df.columns if c.endswith('Population')]
df_long = df.melt(id_vars=['CCA3', 'Country', 'Continent'], value_vars=year_cols,
                  var_name='Year', value_name='Population')

df_long['Year'] = df_long['Year'].str[:4].astype(int)     # '2022 Population' → 2022

print(df.shape, df_long.shape)


(234, 16) (1872, 5)


---
## Q1 — Which continent is growing fastest?

How has the population of each continent **grown since 1970, relative to its 1970 size** — and which continent is growing fastest?


**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*YOUR ANSWER HERE* : When I consider the question, the data and the ideas involved, I would choose a line chart, as I think the question is looking for a comparison of a number of years. Simply seeing the trend of population change of each continent in comparison to itself is the most direct approach.

**Important calculation:** Set 1970 equal to an index of 100 for every continent.

In [16]:
# Q1 - Which continent is growing fastest?

# Calculate total population by continent and year
continent_pop = (
    df_long.groupby(['Continent', 'Year'])['Population']
    .sum()
    .reset_index()
)

# Calculate population index relative to 1970
base_population = (
    continent_pop[continent_pop['Year'] == 1970]
    [['Continent', 'Population']]
    .rename(columns={'Population': 'Population_1970'})
)

continent_pop = continent_pop.merge(
    base_population, on='Continent'
)

continent_pop['Growth Index'] = (
    continent_pop['Population'] /
    continent_pop['Population_1970'] * 100
)

# Identify the fastest-growing continent
latest = continent_pop[continent_pop['Year'] == 2022]
fastest = latest.loc[latest['Growth Index'].idxmax(), 'Continent']

fig = px.line(
    continent_pop,
    x='Year',
    y='Growth Index',
    color='Continent',
    markers=True,
    title=f'{fastest} has experienced the fastest relative population growth',
    labels={
        'Growth Index': 'Population index (1970 = 100)',
        'Year': 'Year',
        'Continent': 'Continent'
    },
    color_discrete_sequence=px.colors.qualitative.Safe
)

fig.update_layout(
    template='plotly_white',
    title_x=0.03,
    height=550,
    legend_title_text='Continent',
    hovermode='x unified'
)

fig.update_xaxes(showgrid=False, dtick=10)
fig.update_yaxes(showgrid=True, gridcolor='#EAEAEA')

fig.show()


---
## Q2 — Which countries added the most people?

Which **15 countries added the most people** between 1970 and 2022 — and how many of them are in **Africa**?

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*YOUR ANSWER HERE* : A sorted horizontal bar chart fits the question where it asks for the 15 countries with the largest population increase. The length of the bars shows the absolute number of people added, while sorting makes the ranking of the countries clear.

In [17]:

# Q2 - Which countries added the most people?

# Calculate absolute population increase
df['Population Added'] = (
    df['2022 Population'] - df['1970 Population']
)

# Select the 15 countries with the largest increases
top15 = df.nlargest(15, 'Population Added').copy()

# Count African countries
africa_count = (top15['Continent'] == 'Africa').sum()

# Highlight African countries
top15['Group'] = np.where(
    top15['Continent'] == 'Africa',
    'Africa',
    'Other continents'
)

# Sort for horizontal bar chart
top15 = top15.sort_values('Population Added', ascending=True)

fig = px.bar(
    top15,
    x='Population Added',
    y='Country',
    orientation='h',
    color='Group',
    text=top15['Population Added'].apply(
        lambda x: f'{x/1e6:.1f}M'
    ),
    color_discrete_map={
        'Africa': '#168C83',
        'Other continents': '#B8B8B8'
    },
    title=f'The largest population increases are concentrated in Asia; {africa_count} African countries make the top 15',
    labels={
        'Population Added': 'Additional people (millions)',
        'Country': '',
        'Group': 'Region'
    }
)

fig.update_traces(
    x=top15['Population Added'] / 1e6,
    textposition='outside',
    cliponaxis=False
)

fig.update_layout(
    template='plotly_white',
    title_x=0.03,
    height=650,
    showlegend=True
)

fig.update_xaxes(
    title='Additional people (millions)',
    showgrid=True,
    gridcolor='#EAEAEA'
)

fig.update_yaxes(showgrid=False)

fig.show()

---
## Q3 — Do bigger countries have more people?

Is there a relationship between a country's **land area** and its **2022 population** — does it differ by **continent**, and which countries clearly break the pattern?

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*YOUR ANSWER HERE* : A scatter plot is suitable here as both land area and population are variables with a numerical value. The plot can be used to see if larger land areas are related to larger population, distinguish continents by their colour and flag country that don't follow the trend.
I would graph the results with log axes as the differences in the country values are so large. Without log scales it would be hard to see the differences between the smaller countries.

In [18]:

# Q3 - Do bigger countries have more people?

# Remove invalid values before using logarithmic axes
plot_df = df[
    (df['Area'] > 0) &
    (df['2022 Population'] > 0)
].copy()

# Fit a simple log-log reference relationship
x = np.log10(plot_df['Area'])
y = np.log10(plot_df['2022 Population'])

slope, intercept = np.polyfit(x, y, 1)

plot_df['Residual'] = (
    y - (slope * x + intercept)
)

# Identify countries furthest from the reference relationship
outliers = plot_df.reindex(
    plot_df['Residual'].abs()
    .nlargest(4).index
)

fig = px.scatter(
    plot_df,
    x='Area',
    y='2022 Population',
    color='Continent',
    hover_name='Country',
    hover_data={
        'Area': ':,',
        '2022 Population': ':,',
        'Continent': True
    },
    title='Land area and population show a relationship, but some countries stand out',
    labels={
        'Area': 'Land area (km²)',
        '2022 Population': 'Population (2022)',
        'Continent': 'Continent'
    },
    color_discrete_sequence=px.colors.qualitative.Safe,
    opacity=0.75
)

# Highlight unusual countries
fig.add_trace(
    px.scatter(
        outliers,
        x='Area',
        y='2022 Population'
    ).data[0].update(
        marker=dict(
            color='#D97706',
            size=13,
            symbol='diamond',
            line=dict(width=1, color='black')
        ),
        name='Unusual observations',
        showlegend=True,
        hovertemplate=(
            '<b>%{customdata[0]}</b><br>'
            'Area: %{x:,.0f} km²<br>'
            'Population: %{y:,.0f}<extra></extra>'
        ),
        customdata=outliers[['Country']].values
    )
)

fig.update_layout(
    template='plotly_white',
    title_x=0.03,
    height=600
)

fig.update_xaxes(
    type='log',
    title='Land area (km², logarithmic scale)',
    showgrid=True,
    gridcolor='#EAEAEA'
)

fig.update_yaxes(
    type='log',
    title='Population (logarithmic scale)',
    showgrid=True,
    gridcolor='#EAEAEA'
)

fig.show()

---
## Q4 — How crowded are the continents?

How does **population density vary within and across continents**? Show the spread, not just the average.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*YOUR ANSWER HERE* : A violin plot is a good option for this question, because we are asked to see the distribution of population density both within each continent, and between continents. We can see the shape of each distribution, not just the average density. Including the internal box plot makes the median and interquartile range clear to see.

In [19]:

# Q4 - How crowded are the continents?

fig = px.violin(
    df,
    x='Continent',
    y='Density',
    color='Continent',
    box=True,
    points='outliers',
    title='Population density varies widely within each continent',
    labels={
        'Continent': 'Continent',
        'Density': 'People per km² (2022)'
    },
    color_discrete_sequence=px.colors.qualitative.Safe
)

fig.update_traces(
    meanline_visible=True,
    opacity=0.8
)

fig.update_layout(
    template='plotly_white',
    title_x=0.03,
    height=600,
    showlegend=False
)

# Log scale helps show both low and very high densities
fig.update_yaxes(
    type='log',
    title='Population density (people per km², log scale)',
    showgrid=True,
    gridcolor='#EAEAEA'
)

fig.update_xaxes(showgrid=False)

fig.show()

---
## Q5 — Where did populations grow — and where did they shrink?

Which countries' populations **grew and which shrank between 2000 and 2022** (percentage change), and where in the world are they?

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*YOUR ANSWER HERE* : Choropleth is the most appropriate map choice as the question not only asked which countries gained or lost population, but where they are situated also. Due to the use of colour which indicates the percentage change between 2000 and 2022, it is easy to see at a glance.

In [20]:

# Q5 - Where did populations grow and shrink?

# Calculate percentage population change
df['Population Change (%)'] = (
    (df['2022 Population'] - df['2000 Population']) /
    df['2000 Population']
) * 100

# Symmetric colour range around zero
limit = df['Population Change (%)'].abs().max()

fig = px.choropleth(
    df,
    locations='CCA3',
    locationmode='ISO-3',
    color='Population Change (%)',
    hover_name='Country',
    hover_data={
        'CCA3': False,
        'Population Change (%)': ':.1f',
        '2000 Population': ':,',
        '2022 Population': ':,'
    },
    color_continuous_scale=[
        '#6A3D9A',
        '#F7F7F7',
        '#168C83'
    ],
    color_continuous_midpoint=0,
    range_color=[-150, 150],
    title='Population growth and decline reveal contrasting patterns across countries',
    labels={
        'Population Change (%)': 'Population change (%)'
    }
)

fig.update_layout(
    template='plotly_white',
    title_x=0.03,
    height=650,
    margin=dict(l=10, r=10, t=80, b=10)
)

fig.update_geos(
    showframe=False,
    showcoastlines=True,
    coastlinecolor='#AAAAAA',
    showland=True,
    landcolor='#F2F2F2',
    showcountries=True,
    countrycolor='white',
    projection_type='natural earth'
)

fig.show()

---
## Q6 — Where do the world's 8 billion people live?

How is the **2022 world population** distributed across **continents and the countries within them**? Show both levels in one figure.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*YOUR ANSWER HERE* : A Treemap is suitable for the question because it deals with nested data structures. You can see that each continent is a large group, whereas the countries are small groups. You can easily see which country contributed how much to the population for each of the continents.

In [21]:

# Q6 - Where do the world's 8 billion people live?

# Create hierarchical treemap
fig = px.treemap(
    df,
    path=[
        px.Constant('World'),
        'Continent',
        'Country'
    ],
    values='2022 Population',
    color='Continent',
    hover_data={
        '2022 Population': ':,'
    },
    color_discrete_sequence=px.colors.qualitative.Safe,
    title='Asia contains the largest share of the world population, followed by Africa'
)

fig.update_traces(
    textinfo='label+percent parent',
    textfont_size=12,
    marker=dict(
        line=dict(
            color='white',
            width=2
        )
    )
)

fig.update_layout(
    template='plotly_white',
    title_x=0.03,
    height=700,
    margin=dict(l=10, r=10, t=80, b=10)
)

fig.show()

---
## Q7 — Whose share of the world is rising?

How have the shares of world population held by **Africa, Asia and Europe** changed from 1970 to 2022? Group the other three continents as `'Rest of the world'`.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*YOUR ANSWER HERE* : The 100% stacked bar chart is appropriate type of graph to use because the question asks how the various continental groups are contributing to total world population over time. Each of the bars shows 100% of the world's population in one year, so it is possible to compare proportions of Africa, Asia, and Europe and the rest of the world over time.

In [22]:
# Q7 - Whose share of the world is rising?

# Create a copy of the long-format data
share_df = df_long.copy()

# Group North America, South America and Oceania
share_df['Group'] = share_df['Continent'].replace({
    'North America': 'Rest of the world',
    'South America': 'Rest of the world',
    'Oceania': 'Rest of the world'
})

# Aggregate population for the four groups
group_population = (
    share_df.groupby(['Year', 'Group'])['Population']
    .sum()
    .reset_index()
)

# Calculate total world population for each year
world_total = (
    group_population.groupby('Year')['Population']
    .sum()
    .reset_index(name='World Population')
)

group_population = group_population.merge(
    world_total,
    on='Year'
)

# Calculate percentage share of world population
group_population['World Share (%)'] = (
    group_population['Population'] /
    group_population['World Population'] * 100
)

# Define meaningful colours
colour_map = {
    'Africa': '#168C83',
    'Asia': '#4776A8',
    'Europe': '#D4A64A',
    'Rest of the world': '#B8B8B8'
}

fig = px.bar(
    group_population,
    x='Year',
    y='World Share (%)',
    color='Group',
    barmode='stack',
    title='Africa is gaining share of the world population while Europe is losing ground',
    labels={
        'Year': 'Year',
        'World Share (%)': 'Share of world population (%)',
        'Group': 'Region'
    },
    color_discrete_map=colour_map,
    category_orders={
        'Group': [
            'Africa',
            'Asia',
            'Europe',
            'Rest of the world'
        ]
    }
)

fig.update_layout(
    template='plotly_white',
    title_x=0.03,
    height=600,
    yaxis=dict(
        range=[0, 100],
        ticksuffix='%',
        showgrid=True,
        gridcolor='#EAEAEA'
    ),
    xaxis=dict(
        showgrid=False,
        dtick=10
    ),
    legend_title_text='Population group',
    hovermode='x unified'
)

fig.update_traces(
    marker_line_width=0
)

fig.show()